# Plotting Basics - Problem Set Solutions

Worked answers to `hist_mplhep_problem_set.ipynb`. These are *one* way to solve each problem, not the only way &mdash; anything that produces the same plot is fine.

In [ ]:
import itertools

import numpy as np
import matplotlib.pyplot as plt

import hist
import mplhep

## Ch. 1 Solutions

### 1.1 - Pros and cons of the `Axes` API vs the `pyplot` API

**`pyplot` (stateful, implicit)**

* *Pros:* least typing; no objects to keep track of; ideal for a quick interactive look at an array.
* *Cons:* commands act on a hidden "current" figure, so it is ambiguous which plot you are modifying; it breaks down with several open figures; state leaks between notebook cells that are run out of order; awkward to wrap in a function that should draw into a caller-supplied plot.

**`Axes` (object-oriented, explicit)**

* *Pros:* you always know which `Axes` you are drawing on; required for multi-panel figures (`fig, axs = plt.subplots(1, 2)`); functions can take an `ax=` argument and compose &mdash; which is exactly how every `mplhep` plotting function is designed; reproducible when cells are re-run.
* *Cons:* more verbose, and you have to carry the `fig`/`ax` handles around.

**When to use each:** `pyplot` for a throwaway look at data; the `Axes` API for anything that goes into a script, a paper figure, or has more than one panel.

### 1.2 - A bar plot with `.bar()`

In [ ]:
positions = [1, 2, 3]
bin_values = [2, 3, 4]

fig, ax = plt.subplots()
ax.bar(positions, bin_values)
ax.set_title("Bar plot with the Axes API")
ax.set_xlabel("Position")
ax.set_ylabel("Counts");

The same thing with the `pyplot` API, for comparison:

In [ ]:
plt.bar(positions, bin_values)
plt.title("Bar plot with the pyplot API")
plt.xlabel("Position")
plt.ylabel("Counts");

### 1.3 - The same numbers as a step histogram

`stairs` takes bin contents plus (optionally) the bin edges &mdash; one more edge than there are values. `baseline=0` with `fill=True` gives the filled look.

In [ ]:
bin_values = [2, 3, 4]
edges = [0.5, 1.5, 2.5, 3.5]
other_values = [3, 2, 2]

fig, ax = plt.subplots()
ax.stairs(bin_values, edges, baseline=0, fill=True, alpha=0.6, label="A")
ax.stairs(other_values, edges, ls="--", color="black", label="B")
ax.set_xlabel("Position")
ax.set_ylabel("Counts")
ax.legend();

## Ch. 2 Solutions

### 2.1 - A `uproot` histogram, styled

`uproot`'s histogram objects follow UHI, so they go straight into `mplhep.histplot`. "Styling you would show at a group meeting" means: an experiment style, uncertainties, axis labels, and a label block.

In [ ]:
import uproot
from skhep_testdata import data_path

file_name = data_path("uproot-hepdata-example.root")
root_file = uproot.open(file_name)
histogram = root_file["hpx"]

mplhep.style.use(mplhep.style.ATLAS)

fig, ax = plt.subplots()
mplhep.histplot(histogram, yerr=True, histtype="fill", label="hpx", ax=ax)
ax.set_xlabel("x")
ax.set_ylabel("Events")
ax.legend(loc="upper right")
mplhep.atlas.label("Internal", data=False, loc=4, ax=ax);

If you want to manipulate the histogram rather than only draw it, convert it first &mdash; `.to_hist()` gives you everything from Ch. 3:

In [ ]:
converted = histogram.to_hist()
converted[10:90].plot();

### 2.2 - The same for a PyROOT histogram

Recent ROOT versions implement UHI as well, so nothing changes except where the object came from.

In [ ]:
import ROOT

histogram = ROOT.TH1F("h1", "h1", 50, -2.5, 2.5)
histogram.FillRandom("gaus", 10000)

fig, ax = plt.subplots()
mplhep.histplot(histogram, yerr=True, histtype="fill", label="Gaussian", ax=ax)
ax.set_xlabel("x")
ax.set_ylabel("Events")
ax.legend(loc="upper right")
mplhep.atlas.label("Internal", data=False, loc=4, ax=ax);

### 2.3 - A Data/MC comparison

`stack=True` with `sort="yield"` orders and piles the MC; the data goes on top as black points with `histtype="errorbar"`.

In [ ]:
bins = np.linspace(0, 10, 21)
x = 0.5 * (bins[1:] + bins[:-1])
mc = [
    120 * np.exp(-x / 2),                 # falling
    60 * np.exp(-x / 5),                  # flatter
    40 * np.exp(-((x - 6) ** 2) / 2),     # peaking
]
labels = ["Z+jets", "Top", "Diboson"]
data = np.random.poisson(np.sum(mc, axis=0))

fig, ax = plt.subplots()
mplhep.histplot(
    mc, bins=bins, stack=True, histtype="fill", sort="yield", label=labels, ax=ax
)
mplhep.histplot(
    data, bins=bins, yerr=True, histtype="errorbar", color="black", label="Data", ax=ax
)
ax.set_xlabel("Observable")
ax.set_ylabel("Events")
ax.legend()
mplhep.atlas.label("Internal", data=True, lumi=140, loc=4, ax=ax);

## Ch. 3 Solutions

### 3.1 - Filling a two-axis histogram

`.fill()` takes one array per axis. Passing them by name (`x=`, `y=`) is safer than by position because it does not depend on the axis order.

In [ ]:
two_axes_hist = hist.Hist(
    hist.axis.Regular(10, 0, 10, name="x", label="x-axis"),
    hist.axis.Variable([0, 1, 2, 5, 10], name="y", label="y-axis"),
    hist.storage.Int64(),
)

two_axes_hist.fill(
    x=np.random.normal(5, 2, 10000),
    y=np.random.uniform(0, 10, 10000),
)

two_axes_hist

A 2D histogram draws as a heatmap, and projecting gives back the 1D distributions:

In [ ]:
two_axes_hist.plot2d();

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
two_axes_hist.project("x").plot(ax=axs[0])
two_axes_hist.project("y").plot(ax=axs[1]);

### 3.2 - A histogram that `h.fill(["Electron", "Muon", "Muon"])` works on

The fill passes *strings*, so the axis has to be a `StrCategory`. Listing the categories up front works:

In [ ]:
h = hist.Hist(hist.axis.StrCategory(["Electron", "Muon"], name="particle"))

h.fill(["Electron", "Muon", "Muon"])
h

and so does starting empty with `growth=True`, which is what you want when you do not know the categories in advance:

In [ ]:
h = hist.new.StrCat([], growth=True, name="particle").Int64()

h.fill(["Electron", "Muon", "Muon"])
h

In [ ]:
h.plot();

## Final Exam Solution

Setup, repeated from the problem set:

In [ ]:
nd_hist = (
    hist.new.Reg(100, 0, 100, name="x", label="Observable")
    .Var([0, 0.2, 0.5, 0.9, 1], name="tag", label="Some MVA")
    .StrCat(["A"], growth=True, name="dataset")
    .IntCat([0, 1, 2, 3], name="region")
    .StrCat(["A"], growth=True, name="syst", label="Systematic")
    .Weight()
)


# Small random letter helper
def rnd_letters(a="A", z="Z", N=10):
    A, Z = np.array([a, z]).view("int32")
    return list(
        np.random.randint(low=A, high=Z, size=N, dtype="int32").view(f"U{N}")[0]
    )


N = 400000
for sample in set(rnd_letters("A", "G", 500)):
    nd_hist.fill(
        x=np.random.normal(np.random.randint(20, 80, 1), 10, N),
        tag=np.random.uniform(0, 1, N),
        dataset=sample,
        region=np.random.randint(0, 4, N),
        syst=rnd_letters("P", "Z", N=N),
    )

nd_hist

**Step 1 & 2 - one 1D distribution per dataset, and its mean.**

Selecting a single category by name and passing `sum` for every other axis collapses the five dimensional histogram down to `x` only. The mean then comes from the bin contents and bin centers.

In [ ]:
datasets = list(nd_hist.axes["dataset"])

projections = {}
means = {}
for dataset in datasets:
    projection = nd_hist[{"dataset": dataset, "tag": sum, "region": sum, "syst": sum}]
    projections[dataset] = projection
    means[dataset] = np.average(projection.axes["x"].centers, weights=projection.values())

for dataset, mean in sorted(means.items(), key=lambda item: item[1]):
    print(f"dataset {dataset}: mean(x) = {mean:.2f}")

The means are spread over tens of units while each distribution has a width of about 10, so yes &mdash; the datasets are clearly separated.

**Step 3 - the two most separated datasets.**

In [ ]:
first, second = max(
    itertools.combinations(datasets, 2),
    key=lambda pair: abs(means[pair[0]] - means[pair[1]]),
)

print(f"Most separated: {first} and {second}")
print(f"Separation: {abs(means[first] - means[second]):.2f}")

**Step 4 - overlay them with ATLAS Internal styling.**

`density=True` normalizes both to unit area, so datasets with different numbers of entries can still be compared by shape.

In [ ]:
mplhep.style.use(mplhep.style.ATLAS)

fig, ax = plt.subplots()
mplhep.histplot(
    [projections[first], projections[second]],
    density=True,
    yerr=True,
    label=[f"Dataset {first}", f"Dataset {second}"],
    ax=ax,
)
ax.set_xlabel("Observable")
ax.set_ylabel("Fraction of events / bin")
ax.legend()
mplhep.atlas.label("Internal", data=False, loc=4, ax=ax);

#### A couple of variations worth knowing

The 100 bins are finer than the structure needs. `hist.rebin` inside a `Slicer` merges them, and the same selection can be restricted to a signal-like region &mdash; here a high MVA score and one region &mdash; which is what you would actually do in an analysis:

In [ ]:
s = hist.tag.Slicer()

fig, ax = plt.subplots()
for dataset in (first, second):
    selection = nd_hist[
        {
            "dataset": dataset,
            "tag": s[0.5j::sum],
            "region": 0,
            "syst": sum,
            "x": s[:: hist.rebin(5)],
        }
    ]
    mplhep.histplot(selection, density=True, yerr=True, label=f"Dataset {dataset}", ax=ax)

ax.set_xlabel("Observable")
ax.set_ylabel("Fraction of events / bin")
ax.legend()
mplhep.atlas.label("Internal", data=False, loc=4, ax=ax);

And `.stack()` on the dataset axis draws every dataset at once, which is the quickest way to eyeball whether *any* of them separate:

In [ ]:
fig, ax = plt.subplots()
nd_hist[{"tag": sum, "region": sum, "syst": sum, "x": s[:: hist.rebin(5)]}].stack(
    "dataset"
).plot(ax=ax)
ax.set_ylabel("Events")
ax.legend(ncol=2)
mplhep.atlas.label("Internal", data=False, loc=4, ax=ax);